## *For up-to-date plotting script use plots/combineFiles.ipynb*

In [ ]:
import pandas as pd
import numpy as np
from coffea import util
import itertools
import os, sys
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
import hist
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
hep.style.use("CMS")

sys.path.append('../python/')
from functions import loadCoffeaFile, getLabelMap, getCoffeaFilenames, plotBackgroundEstimate, getHist


## Scale factors and IOV

In [ ]:
IOVs = ['2016APV', '2016', '2016all', '2017', '2018', 'Full']
# IOVs = ['2016all']

variable = 'jetphi'  # Change this at will

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530./10.,
    "2018": 59800./10., #59740./10., #Blinding
    "Full": 46053. # 137190. Blinded
}

t_BR = 0.6741
ttbar_BR = 0.4544 #PDG 2019
ttbar_xs1 = 831.76 * (0.09210) #pb For ttbar mass from 700 to 1000
ttbar_xs2 = 831.76 * (0.02474) #pb For ttbar mass from 1000 to Inf
toptag_sf = 0.9
toptag_kf = 1.0 #0.7
qcd_xs = 1370000000.0 #pb From https://cms-gen-dev.cern.ch/xsdb



## analysis categories

In [ ]:
# analysis categories #

label_dict =  getLabelMap()
label_to_int_dict = {label: i for i, label in label_dict.items()}

signal_cats = [ i for i, label in label_dict.items() if '2t' in label]
pretag_cats = [ i for i, label in label_dict.items() if 'pre' in label]
anti_cats   = [ i for i, label in label_dict.items() if 'at' in label]

## make plot image filenames

In [ ]:
directories = [
    'images/png/closureTest/2016all',
    'images/png/closureTest/2016APV',
    'images/png/closureTest/2016',
    'images/png/closureTest/2017',
    'images/png/closureTest/2018',
    'images/png/closureTest/Full',
    'images/pdf/closureTest/2016all',
    'images/pdf/closureTest/2016APV',
    'images/pdf/closureTest/2016',
    'images/pdf/closureTest/2017',
    'images/pdf/closureTest/2018',
    'images/pdf/closureTest/Full',
    'images/png/kinematics/2016all',
    'images/png/kinematics/2016APV',
    'images/png/kinematics/2016',
    'images/png/kinematics/2017',
    'images/png/kinematics/2018',
    'images/png/kinematics/Full',
    'images/pdf/kinematics/2016all',
    'images/pdf/kinematics/2016APV',
    'images/pdf/kinematics/2016',
    'images/pdf/kinematics/2017',
    'images/pdf/kinematics/2018',
    'images/pdf/kinematics/Full'
]


for path in directories:
    if not os.path.exists(path):
        os.makedirs(path)

## functions

In [ ]:
def make_error_boxes(ax, xdata, ydata, xerror, yerror, facecolor='none',
                     edgecolor='none', alpha=0.5):
    
    # Loop over data points; create box from errors at each point
    errorboxes = [Rectangle((x - xe, y - ye), xe.sum(), ye.sum()) for x, y, xe, ye in zip(xdata, ydata, xerror.T, yerror.T)]

    # Create patch collection with specified colour/alpha
    pc = PatchCollection(errorboxes, facecolor=facecolor, alpha=alpha,
                         edgecolor=edgecolor)

    # Add collection to axes
    ax.add_collection(pc)

    # Plot errorbars
    artists = ax.errorbar(xdata, ydata, xerr=xerror, yerr=yerror,
                          fmt='none', ecolor='k', barsabove=True)

    return artists

def getHist(hname, ds, bkgest, year, sum_axes=[], integrate_axes={}, masspoint=''):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames()
    
    cfiles = []
    sf = []
    bkgest_str = np.where([bkgest], 'weighted', 'unweighted')[0]
    
    for key, file in coffeaFiles[ds][bkgest_str][year].items():
        if masspoint != '':
            if masspoint in key:
                loaded_file = util.load(file)
                sum_axes_dict = {ax:sum for ax in sum_axes}
                histo = loaded_file[hname][integrate_axes][sum_axes_dict]
                histo = histo * (lumi[IOV] * 1.0 / loaded_file['cutflow']['sumw'])
                return histo
            
        loaded_file = util.load(file)
        cfiles.append(loaded_file)
        
        
        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 * toptag_sf**2 / loaded_file['cutflow']['sumw'])
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 * toptag_sf**2 / loaded_file['cutflow']['sumw'])     
        elif 'QCD' in ds:
            sf.append(lumi[IOV] * qcd_xs / loaded_file['cutflow']['sumw'])  
        else:
            sf.append(1.)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])    
    
    # sum all hists from dataset eras or pt bins
    histo = hists[0]*sf[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
            
            
    return histo
    
            
            
def plotBackgroundEstimate(histname, Hpretag, Hantitag_data, Hantitag_ttbar, Hdata, Hntmj, Httbar, Year, Text='', signame='', xaxis='', linear=False, Pull=False, hsig1=None, hsig2=None, hsig3=None, hsig4=None, hsig5=None):
    
    Hbkg = Hntmj + Httbar
    Ndenom = np.abs(Hantitag_data.values() - Hantitag_ttbar.values())
    mistag = Hntmj / Hpretag.values()
    term1 = np.ones(len(Hpretag.values()))/Hpretag.values()
    term2 = (np.ones(len(mistag.values()))-mistag.values())/(Ndenom*mistag.values())
    errNTMJ = Hntmj.values()*np.sqrt( term1 + term2 )
    errs = np.sqrt(errNTMJ**2 + Httbar.values())
    height = errs * 2
    bottom = Hbkg.values() - errs
    edges = Hntmj.axes[xaxis].edges
    
    fig, (ax1, ax2) = plt.subplots(nrows=2, height_ratios=[3, 1])

    if linear:
        hep.cms.label(Text, data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=0, fontsize=15, ax=ax1)
    else:
        hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=2, fontsize=20, ax=ax1)
        hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)
    
    S = hist.Stack(Httbar, Hntmj)
    S.plot(ax=ax1, stack=True, histtype="fill", color=['xkcd:deep red', 'xkcd:pale gold'], label=['TTbar', 'NTMJ'])
    hep.histplot(Hdata,  ax=ax1, histtype='errorbar', color='black', label='Data')    
#     ax1.errorbar(x=edges[:-1], y=Hbkg.values(), xerr=np.diff(edges), yerr=np.sqrt(Hbkg.values()))
#     make_error_boxes(ax1, edges[:-1], Hbkg.values(), np.diff(edges), np.sqrt(Hbkg.values()))
#     hep.histplot(Hbkg,   ax=ax1, histtype='fill', color='xkcd:pale gold', label='NTMJ')
#     hep.histplot(Httbar, ax=ax1, histtype='fill', color='xkcd:deep red', label='TTbar')

    ax1.bar(x = edges[:-1],
           height=height,
           bottom=bottom,
           width = np.diff(edges), align='edge', hatch='///', edgecolor='gray',
           linewidth=1, facecolor='none', alpha=0.8,
           zorder=10, label='Stat. Unc.')
    
    Signal = {
        'RSGluon' : r'RS$_{KK}$ Gluon',
        'ZPrime1' : r'Z ` $1\%$',
        'ZPrime10': r'Z ` $10\%$',
        'ZPrime30': r'Z ` $30\%$',
        'ZPrimeDM': r'Z ` DM'
    }
    
        
    if hsig1 != None and Pull == True:
        hep.histplot(hsig1, ax=ax1, histtype='step', label=Signal[signame]+' 1 TeV')
        hep.histplot(hsig2, ax=ax1, histtype='step', label=Signal[signame]+' 2 TeV')
        hep.histplot(hsig3, ax=ax1, histtype='step', label=Signal[signame]+' 3 TeV')
        hep.histplot(hsig4, ax=ax1, histtype='step', label=Signal[signame]+' 4 TeV')
        if hsig5 != None:
            hep.histplot(hsig5, ax=ax1, histtype='step', label=Signal[signame]+' 5 TeV')
            
    if Pull != True:
        ratio_plot =  Hdata / Hbkg.values()
        ratioUnc = 1. / errs

        ax2.bar(x = edges[:-1],
               height=(2.*ratioUnc),
               bottom=(np.ones_like(ratio_plot.values()) - ratioUnc),
               width = np.diff(edges), align='edge', edgecolor='gray',
               linewidth=0, facecolor='gray', alpha=0.3,
               zorder=10, label='Unc.')
        hep.histplot(ratio_plot, ax=ax2, histtype='errorbar', color='black')
    
        ax2.set_ylim(0,2)
        ax2.axhline(1, color='black', ls='--')
        ax2.set_ylabel('Data/Bkg')
    elif Pull == True:
        pull_plot = (Hdata + -1*Hbkg) / (np.sqrt(Hdata.values()) + errs)
#         oneSigma   = np.ones_like(pull_plot.values())*0.341
#         twoSigma   = np.ones_like(pull_plot.values())*(0.341 + 0.136)
        
#         ax2.fill_between(np.diff(edges), -twoSigma, twoSigma, alpha=0.2, color='grey')
        
        hep.histplot(pull_plot, ax=ax2, histtype='fill', color='steelblue')
        
        ax2.set_ylim(-3,3)
        ax2.set_yticks([-2,-1,0,1,2])
        ax2.axhline(0, color='black', ls='--')
        ax2.set_ylabel(r'(D-B)/$\sigma$')

    ax1.legend(fontsize='xx-small', loc=1)
    ax1.set_yscale('log')
    ax1.set_ylabel('Events')
    ax1.set_xlabel('')
    ax1.set_ylim(1e-2, 1e7)
    ax1.set_xlim(900, 8000)
    ax2.set_xlim(900, 8000)    
    if linear:
        ax1.set_yscale('linear')
        ax1.autoscale('y')
        ax1.set_xlim(900, 8000)
#         ax1.set_ylim(1e-2, 1e5)
        
    if histname == 'jetpt':
        ax1.set_xlim(400, 2000)
        ax2.set_xlim(400, 2000)
    elif histname == 'jeteta':
        ax1.set_xlim(-2.4, 2.4)
        ax2.set_xlim(-2.4, 2.4)
    elif histname == 'jetphi':
        ax1.set_xlim(-np.pi, np.pi)
        ax2.set_xlim(-np.pi, np.pi)
    elif histname == 'sdjetmass':
        ax1.set_xlim(0, 500)
        ax2.set_xlim(0, 500)   
        
        
        
def plotBackgroundEstimateNoData(histname, Hntmj, Httbar, Year, Text='', signame='', hsig1=None, hsig2=None, hsig3=None, hsig4=None):
    
    Hbkg = Hntmj + Httbar
    
    hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=2, fontsize=20)
    hep.cms.text(Text, loc=2, fontsize=20)

    hep.histplot(Hbkg, histtype='fill', color='xkcd:pale gold', label='NTMJ')
    hep.histplot(Httbar, histtype='fill', color='xkcd:deep red', label='TTbar')
    
    Signal = {
        'RSGluon' : r'RS$_{KK}$ Gluon',
        'ZPrime1' : r'Z ` $1\%$',
        'ZPrime10': r'Z ` $10\%$',
        'ZPrime30': r'Z ` $30\%$',
        'ZPrimeDM': r'Z ` DM'
    }
        
    if hsig1 != None:
        hep.histplot(hsig1, histtype='step', label=Signal['ZPrime1']+' 3 TeV')
        hep.histplot(hsig2, histtype='step', label=Signal['ZPrime10']+' 3 TeV')
        hep.histplot(hsig3, histtype='step', label=Signal['ZPrime30']+' 3 TeV')
        hep.histplot(hsig4, histtype='step', label=Signal['ZPrimeDM']+' 3 TeV')

    plt.legend(fontsize='xx-small')
    plt.yscale('log')
    plt.ylabel('Events')
    plt.xlabel('')
    plt.ylim(1e-2, 1e6)
    plt.xlim(900, 6000)
    plt.xlim(900, 6000)    
    

In [ ]:
dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'

## plot background estimate (inclusive)

In [ ]:
# analysis categories #
label_dict = util.load(f'../outputs/QCD_2016.coffea')['analysisCategories']
label_to_int_dict = {label: i for i, label in label_dict.items()}

signal_cats = [ i for label, i in label_to_int_dict.items() if '2t' in label]
pretag_cats = [ i for label, i in label_to_int_dict.items() if 'pre' in label]
anti_cats   = [ i for label, i in label_to_int_dict.items() if 'at' in label]

In [ ]:
signal = 'ZPrime1'
usePull = True
Linear = True
linearStr = ''
if Linear:
    linearStr = '_LINEAR'
    
if variable == 'ttbarmass':
    
    for IOV in IOVs:
        
        if 'Full' in IOV:

            httbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_apv = getHist('ttbarmass', 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_apv = getHist('ttbarmass', 'JetHT', True, '2016APV',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            httbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_noapv = getHist('ttbarmass', 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_noapv = getHist('ttbarmass', 'JetHT', True, '2016',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_noapv = getHist('ttbarmass', 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            httbar_17 = getHist('ttbarmass', 'TTbar', False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_17 = getHist('ttbarmass', 'TTbar', True, '2017', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_17 = getHist('ttbarmass', 'JetHT', True, '2017',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_17 = getHist('ttbarmass', 'JetHT', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_17 = getHist('ttbarmass', 'JetHT', False, '2017',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_17 = getHist('ttbarmass', 'JetHT', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_17 = getHist('ttbarmass', 'TTbar', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            httbar_18 = getHist('ttbarmass', 'TTbar', False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_18 = getHist('ttbarmass', 'TTbar', True, '2018', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_18 = getHist('ttbarmass', 'JetHT', True, '2018',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_18 = getHist('ttbarmass', 'JetHT', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_18 = getHist('ttbarmass', 'JetHT', False, '2018',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_18 = getHist('ttbarmass', 'JetHT', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_18 = getHist('ttbarmass', 'TTbar', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            hsignal1000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            hsignal1000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            hsignal1000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            hsignal1000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')
                hsignal5000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')
                hsignal5000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')
                hsignal5000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')

            httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
            hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18
            hntmj   = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
            hdata   = hdata_apv + hdata_noapv + hdata_17 + hdata_18
            hpretag = hpretag_apv + hpretag_noapv + hpretag_17 + hpretag_18
            hantitag_data = hantitag_data_apv + hantitag_data_noapv + hantitag_data_17 + hantitag_data_18
            hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv + hantitag_ttbar_17 + hantitag_ttbar_18
            hsignal1000 = hsignal1000_apv + hsignal1000_noapv + hsignal1000_17 + hsignal1000_18
            hsignal2000 = hsignal2000_apv + hsignal2000_noapv + hsignal2000_17 + hsignal2000_18
            hsignal3000 = hsignal3000_apv + hsignal3000_noapv + hsignal3000_17 + hsignal3000_18
            hsignal4000 = hsignal4000_apv + hsignal4000_noapv + hsignal4000_17 + hsignal4000_18
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = hsignal5000_apv + hsignal5000_noapv + hsignal5000_17 + hsignal5000_18

            hntmj_fixed = hntmj + -1*hcontam

        elif '2016all' in IOV:

            httbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_apv = getHist('ttbarmass', 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_apv = getHist('ttbarmass', 'JetHT', True, '2016APV',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            httbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam_noapv = getHist('ttbarmass', 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj_noapv = getHist('ttbarmass', 'JetHT', True, '2016',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag_noapv = getHist('ttbarmass', 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            hsignal1000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            hsignal1000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')
                hsignal5000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')
                

            httbar  = httbar_apv + httbar_noapv
            hcontam = hcontam_apv + hcontam_noapv
            hntmj   = hntmj_apv + hntmj_noapv
            hdata   = hdata_apv + hdata_noapv
            hpretag = hpretag_apv + hpretag_noapv
            hantitag_data = hantitag_data_apv + hantitag_data_noapv
            hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv
            hsignal1000 = hsignal1000_apv + hsignal1000_noapv
            hsignal2000 = hsignal2000_apv + hsignal2000_noapv
            hsignal3000 = hsignal3000_apv + hsignal3000_noapv
            hsignal4000 = hsignal4000_apv + hsignal4000_noapv
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = hsignal5000_apv + hsignal5000_noapv

            hntmj_fixed = hntmj + -1*hcontam


        else:

            httbar  = getHist(variable, 'TTbar', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hntmj   = getHist(variable, 'JetHT', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hdata   = getHist(variable, 'JetHT', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            hpretag = getHist(variable, 'JetHT', False, IOV,sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
            hantitag_data = getHist(variable, 'JetHT', False, IOV,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            hantitag_ttbar = getHist(variable, 'TTbar', False, IOV,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats, 'systematic':'nominal'})
            
            hsignal1000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='1000')
            hsignal2000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
            hsignal3000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='3000')
            hsignal4000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='4000')
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='5000')

            hntmj_fixed = hntmj + -1*hcontam

        if Linear:
#             text = f'Work in Progress'
            text = f'Preliminary'
        else:
#             text = f'data/simulation\nWork in Progress'
            text = f'Preliminary\n'

        if signal == 'RSGluon' or signal == 'ZPrimeDM':
            plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000, hsignal5000)
        else:
            plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
        if usePull:
            savefilename = f'images/png/{dirname}/{IOV}/closuretest_with_{signal}_Inclusive{linearStr}_NoMassMod.png'
        else:
            savefilename = f'images/png/{dirname}/{IOV}/closuretest_Inclusive{linearStr}_NoMassMod.png'
            
        print(savefilename)
#         plt.savefig(savefilename)
#         plt.savefig(savefilename.replace('png', 'pdf'))

        plt.show()
            
else:

    for IOV in IOVs:
        
        if 'Full' in IOV:
            
            httbar_apv = getHist(variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_apv = getHist(variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_apv = getHist(variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_apv = getHist(variable, 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_apv = getHist(variable, 'TTbar', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})

            httbar_noapv = getHist(variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_noapv = getHist(variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_noapv = getHist(variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_noapv = getHist(variable, 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_noapv = getHist(variable, 'TTbar', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            
            httbar_17 = getHist(variable, 'TTbar', False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_17 = getHist(variable, 'TTbar', True, '2017', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_17 = getHist(variable, 'JetHT', True, '2017',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_17 = getHist(variable, 'JetHT', False, '2017',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_17 = getHist(variable, 'TTbar', False, '2017',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            
            httbar_18 = getHist(variable, 'TTbar', False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_18 = getHist(variable, 'TTbar', True, '2018', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_18 = getHist(variable, 'JetHT', True, '2018',   sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_18 = getHist(variable, 'JetHT', False, '2018',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_18 = getHist(variable, 'TTbar', False, '2018',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            
            hsignal1000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            hsignal1000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            hsignal1000_17 = getHist(variable, signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_17 = getHist(variable, signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_17 = getHist(variable, signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_17 = getHist(variable, signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            hsignal1000_18 = getHist(variable, signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_18 = getHist(variable, signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_18 = getHist(variable, signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_18 = getHist(variable, signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
                hsignal5000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
                hsignal5000_17 = getHist(variable, signal, False, '2017', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
                hsignal5000_18 = getHist(variable, signal, False, '2018', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
            
            httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
            hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18
            hntmj   = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
            hdata   = hdata_apv + hdata_noapv + hdata_17 + hdata_18
            hpretag = hpretag_apv + hpretag_noapv + hpretag_17 + hpretag_18
            hantitag_data = hantitag_data_apv + hantitag_data_noapv + hantitag_data_17 + hantitag_data_18
            hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv + hantitag_ttbar_17 + hantitag_ttbar_18
            hsignal1000 = hsignal1000_apv + hsignal1000_noapv + hsignal1000_17 + hsignal1000_18
            hsignal2000 = hsignal2000_apv + hsignal2000_noapv + hsignal2000_17 + hsignal2000_18
            hsignal3000 = hsignal3000_apv + hsignal3000_noapv + hsignal3000_17 + hsignal3000_18
            hsignal4000 = hsignal4000_apv + hsignal4000_noapv + hsignal4000_17 + hsignal4000_18
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = hsignal5000_apv + hsignal5000_noapv + hsignal5000_17 + hsignal5000_18
            
            hntmj_fixed = hntmj + -1*hcontam

        elif '2016all' in IOV:

            httbar_apv = getHist(variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_apv = getHist(variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_apv = getHist(variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_apv = getHist(variable, 'JetHT', False, '2016APV',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_apv = getHist(variable, 'TTbar', False, '2016APV',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})

            httbar_noapv = getHist(variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj_noapv = getHist(variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata_noapv = getHist(variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag_noapv = getHist(variable, 'JetHT', False, '2016',sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar_noapv = getHist(variable, 'TTbar', False, '2016',  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            
            hsignal1000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            hsignal1000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')
            
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000_apv = getHist(variable, signal, False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
                hsignal5000_noapv = getHist(variable, signal, False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
            
            httbar  = httbar_apv + httbar_noapv
            hcontam = hcontam_apv + hcontam_noapv
            hntmj   = hntmj_apv + hntmj_noapv
            hdata   = hdata_apv + hdata_noapv
            hpretag = hpretag_apv + hpretag_noapv
            hantitag_data = hantitag_data_apv + hantitag_data_noapv
            hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv
            hsignal1000 = hsignal1000_apv + hsignal1000_noapv
            hsignal2000 = hsignal2000_apv + hsignal2000_noapv
            hsignal3000 = hsignal3000_apv + hsignal3000_noapv
            hsignal4000 = hsignal4000_apv + hsignal4000_noapv
            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = hsignal5000_apv + hsignal5000_noapv

            hntmj_fixed = hntmj + -1*hcontam

        else:

            httbar = getHist(variable, 'TTbar', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hntmj = getHist(variable, 'JetHT', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hdata = getHist(variable, 'JetHT', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
            hpretag = getHist(variable, 'JetHT', False, IOV,sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
            hantitag_data = getHist(variable, 'JetHT', False, IOV,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            hantitag_ttbar = getHist(variable, 'TTbar', False, IOV,  sum_axes=['anacat'], integrate_axes={'anacat':anti_cats})
            
            hsignal1000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='1000')
            hsignal2000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='2000')
            hsignal3000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='3000')
            hsignal4000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='4000')

            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                hsignal5000 = getHist(variable, signal, False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats}, masspoint='5000')
                
            hntmj_fixed = hntmj + -1*hcontam


        if Linear:
#             text = f'Work in Progress'
            text = f'Preliminary'
        else:
#             text = f'data/simulation\nWork in Progress\n'
            text = f'Preliminary\n'

        if signal == 'RSGluon' or signal == 'ZPrimeDM':
            plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000, hsignal5000)
        else:
            plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
        if usePull:
            savefilename = f'images/png/{dirname}/{IOV}/{variable}_with_{signal}_Inclusive{linearStr}.png'
        else:
            savefilename = f'images/png/{dirname}/{IOV}/{variable}_Inclusive{linearStr}.png'
        
        print(savefilename)
#         plt.savefig(savefilename)
#         plt.savefig(savefilename.replace('png', 'pdf'))

        plt.show()

## plot background estimate (by category)

In [ ]:
signal = 'ZPrime1'
usePull = True
Linear = True
if Linear:
    linearStr = '_LINEAR'

cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']

if variable == 'ttbarmass':
    
    for IOV in IOVs:
        
        for cat in cats:
            
            signal_cat = label_to_int_dict['2t'+cat]
            pretag_cat = label_to_int_dict['pret'+cat]
            anti_cat   = label_to_int_dict['at'+cat]
            
            if 'Full' in IOV:

                httbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_apv = getHist('ttbarmass', 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_apv = getHist('ttbarmass', 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                httbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_noapv = getHist('ttbarmass', 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_noapv = getHist('ttbarmass', 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_noapv = getHist('ttbarmass', 'JetHT', False, '2016',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                httbar_17 = getHist('ttbarmass', 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_17 = getHist('ttbarmass', 'TTbar', True, '2017', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_17 = getHist('ttbarmass', 'JetHT', True, '2017',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_17 = getHist('ttbarmass', 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_17 = getHist('ttbarmass', 'JetHT', False, '2017',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_17 = getHist('ttbarmass', 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_17 = getHist('ttbarmass', 'TTbar', False, '2017',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                httbar_18 = getHist('ttbarmass', 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_18 = getHist('ttbarmass', 'TTbar', True, '2018', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_18 = getHist('ttbarmass', 'JetHT', True, '2018',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_18 = getHist('ttbarmass', 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_18 = getHist('ttbarmass', 'JetHT', False, '2018',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_18 = getHist('ttbarmass', 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_18 = getHist('ttbarmass', 'TTbar', False, '2018',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                hsignal1000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                hsignal1000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                hsignal1000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                hsignal1000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')
                    hsignal5000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')
                    hsignal5000_17 = getHist('ttbarmass', signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')
                    hsignal5000_18 = getHist('ttbarmass', signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')

                httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
                hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18
                hntmj   = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
                hdata   = hdata_apv + hdata_noapv + hdata_17 + hdata_18
                hpretag = hpretag_apv + hpretag_noapv + hpretag_17 + hpretag_18
                hantitag_data = hantitag_data_apv + hantitag_data_noapv + hantitag_data_17 + hantitag_data_18
                hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv + hantitag_ttbar_17 + hantitag_ttbar_18
                hsignal1000 = hsignal1000_apv + hsignal1000_noapv + hsignal1000_17 + hsignal1000_18
                hsignal2000 = hsignal2000_apv + hsignal2000_noapv + hsignal2000_17 + hsignal2000_18
                hsignal3000 = hsignal3000_apv + hsignal3000_noapv + hsignal3000_17 + hsignal3000_18
                hsignal4000 = hsignal4000_apv + hsignal4000_noapv + hsignal4000_17 + hsignal4000_18
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = hsignal5000_apv + hsignal5000_noapv + hsignal5000_17 + hsignal5000_18

                hntmj_fixed = hntmj + -1*hcontam
            
            elif '2016all' in IOV:
                
                httbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_apv = getHist('ttbarmass', 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_apv = getHist('ttbarmass', 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                httbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_noapv = getHist('ttbarmass', 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_noapv = getHist('ttbarmass', 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag_noapv = getHist('ttbarmass', 'JetHT', False, '2016',sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                hsignal1000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                hsignal1000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')

                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000_apv = getHist('ttbarmass', signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')
                    hsignal5000_noapv = getHist('ttbarmass', signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')


                httbar  = httbar_apv + httbar_noapv
                hcontam = hcontam_apv + hcontam_noapv
                hntmj   = hntmj_apv + hntmj_noapv
                hdata   = hdata_apv + hdata_noapv
                hpretag = hpretag_apv + hpretag_noapv
                hantitag_data = hantitag_data_apv + hantitag_data_noapv
                hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv
                hsignal1000 = hsignal1000_apv + hsignal1000_noapv
                hsignal2000 = hsignal2000_apv + hsignal2000_noapv
                hsignal3000 = hsignal3000_apv + hsignal3000_noapv
                hsignal4000 = hsignal4000_apv + hsignal4000_noapv
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = hsignal5000_apv + hsignal5000_noapv

                hntmj_fixed = hntmj + -1*hcontam
            

            else: # Pick it up here...

                httbar  = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj   = getHist(variable, 'JetHT', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata   = getHist(variable, 'JetHT', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hpretag = getHist(variable, 'JetHT', False, IOV,sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hantitag_data = getHist(variable, 'JetHT', False, IOV,  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})
                hantitag_ttbar = getHist(variable, 'TTbar', False, IOV,  sum_axes=[], integrate_axes={'anacat':anti_cat, 'systematic':'nominal'})

                hsignal1000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')

                hntmj_fixed = hntmj + -1*hcontam
                        
            dytext = ''
            if 'cen' in cat:
                dytext = r'$\Delta y$ < 1.0'
            elif 'fwd' in cat:
                dytext = r'$\Delta y$ > 1.0'
            
            btext = ''
            if '0b' in cat:
                btext = '0 b-tags'
            elif '1b' in cat:
                btext = '1 b-tag'
            elif '2b' in cat:
                btext = '2 b-tags'
            
            if Linear:
                text = f'Preliminary:    {btext}, {dytext}'
#                 text = f'data/simulation\nWork in Progress\n{btext} {dytext}'
            else:
#                 text = f'data/simulation\nWork in Progress\n{btext} {dytext}'
                text = f'Preliminary\n{btext}, {dytext} \n'

            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000, hsignal5000)
            else:
                plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
                
            
            if usePull:
                savefilename = f'images/png/{dirname}/{IOV}/closuretest_with_{signal}_{cat}{linearStr}.png'
            else:
                savefilename = f'images/png/{dirname}/{IOV}/closuretest_{cat}{linearStr}.png'
            

            print(savefilename)
            plt.savefig(savefilename)
            plt.savefig(savefilename.replace('png', 'pdf'))

            plt.show()

else:

    for IOV in IOVs:

        for cat in cats:

            signal_cat = label_to_int_dict['2t'+cat]
            pretag_cat = label_to_int_dict['pret'+cat]
            anti_cat   = label_to_int_dict['at'+cat]
            
            if 'Full' in IOV:
            
                httbar_apv = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_apv = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_apv = getHist(variable, 'JetHT', False, '2016APV',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_apv = getHist(variable, 'TTbar', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat})

                httbar_noapv = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_noapv = getHist(variable, 'JetHT', False, '2016',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_noapv = getHist(variable, 'TTbar', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat})

                httbar_17 = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_17 = getHist(variable, 'TTbar', True, '2017', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_17 = getHist(variable, 'JetHT', True, '2017',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_17 = getHist(variable, 'JetHT', False, '2017',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_17 = getHist(variable, 'TTbar', False, '2017',  sum_axes=[], integrate_axes={'anacat':anti_cat})

                httbar_18 = getHist(variable, 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_18 = getHist(variable, 'TTbar', True, '2018', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_18 = getHist(variable, 'JetHT', True, '2018',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_18 = getHist(variable, 'JetHT', False, '2018',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_18 = getHist(variable, 'TTbar', False, '2018',  sum_axes=[], integrate_axes={'anacat':anti_cat})

                hsignal1000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')

                hsignal1000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')

                hsignal1000_17 = getHist(variable, signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_17 = getHist(variable, signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_17 = getHist(variable, signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_17 = getHist(variable, signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')

                hsignal1000_18 = getHist(variable, signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_18 = getHist(variable, signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_18 = getHist(variable, signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_18 = getHist(variable, signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')

                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')
                    hsignal5000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')
                    hsignal5000_17 = getHist(variable, signal, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')
                    hsignal5000_18 = getHist(variable, signal, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')

                httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
                hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18
                hntmj   = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
                hdata   = hdata_apv + hdata_noapv + hdata_17 + hdata_18
                hpretag = hpretag_apv + hpretag_noapv + hpretag_17 + hpretag_18
                hantitag_data = hantitag_data_apv + hantitag_data_noapv + hantitag_data_17 + hantitag_data_18
                hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv + hantitag_ttbar_17 + hantitag_ttbar_18
                hsignal1000 = hsignal1000_apv + hsignal1000_noapv + hsignal1000_17 + hsignal1000_18
                hsignal2000 = hsignal2000_apv + hsignal2000_noapv + hsignal2000_17 + hsignal2000_18
                hsignal3000 = hsignal3000_apv + hsignal3000_noapv + hsignal3000_17 + hsignal3000_18
                hsignal4000 = hsignal4000_apv + hsignal4000_noapv + hsignal4000_17 + hsignal4000_18
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = hsignal5000_apv + hsignal5000_noapv + hsignal5000_17 + hsignal5000_18

                hntmj_fixed = hntmj + -1*hcontam

            elif '2016all' in IOV:

                httbar_apv = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_apv = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_apv = getHist(variable, 'JetHT', False, '2016APV',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_apv = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_apv = getHist(variable, 'TTbar', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':anti_cat})

                httbar_noapv = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag_noapv = getHist(variable, 'JetHT', False, '2016',sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar_noapv = getHist(variable, 'TTbar', False, '2016',  sum_axes=[], integrate_axes={'anacat':anti_cat})
                
                hsignal1000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')

                hsignal1000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')
                
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000_apv = getHist(variable, signal, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')
                    hsignal5000_noapv = getHist(variable, signal, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')

                httbar = httbar_apv + httbar_noapv
                hcontam = hcontam_apv + hcontam_noapv
                hntmj = hntmj_apv + hntmj_noapv
                hdata = hdata_apv + hdata_noapv
                hpretag = hpretag_apv + hpretag_noapv
                hantitag_data = hantitag_data_apv + hantitag_data_noapv
                hantitag_ttbar = hantitag_ttbar_apv + hantitag_ttbar_noapv
                hsignal1000 = hsignal1000_apv + hsignal1000_noapv
                hsignal2000 = hsignal2000_apv + hsignal2000_noapv
                hsignal3000 = hsignal3000_apv + hsignal3000_noapv
                hsignal4000 = hsignal4000_apv + hsignal4000_noapv
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = hsignal5000_apv + hsignal5000_noapv
                
                hntmj_fixed = hntmj + -1*hcontam

            else:

                httbar = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj = getHist(variable, 'JetHT', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata = getHist(variable, 'JetHT', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat})
                hpretag = getHist(variable, 'JetHT', False, IOV,sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hantitag_data = getHist(variable, 'JetHT', False, IOV,  sum_axes=[], integrate_axes={'anacat':anti_cat})
                hantitag_ttbar = getHist(variable, 'TTbar', False, IOV,  sum_axes=[], integrate_axes={'anacat':anti_cat})
                
                hsignal1000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='1000')
                hsignal2000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='2000')
                hsignal3000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='3000')
                hsignal4000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='4000')
                
                if signal == 'RSGluon' or signal == 'ZPrimeDM':
                    hsignal5000 = getHist(variable, signal, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat}, masspoint='5000')
                
                hntmj_fixed = hntmj + -1*hcontam

            dytext = ''
            if 'cen' in cat:
                dytext = r'$|\Delta y|$ < 1.0'
            elif 'fwd' in cat:
                dytext = r'$|\Delta y|$ > 1.0'

            btext = ''
            if '0b' in cat:
                btext = '0 b-tags'
            elif '1b' in cat:
                btext = '1 b-tag'
            elif '2b' in cat:
                btext = '2 b-tags'

            if Linear:
                text = f'Preliminary:    {btext}, {dytext}'
#                 text = f'data/simulation\nWork in Progress\n{btext} {dytext}'
            else:
#                 text = f'data/simulation\nWork in Progress\n{btext} {dytext}'
                text = f'Preliminary\n{btext}, {dytext} \n'

            if signal == 'RSGluon' or signal == 'ZPrimeDM':
                plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000, hsignal5000)
            else:
                plotBackgroundEstimate(variable, hpretag, hantitag_data, hantitag_ttbar, hdata, hntmj_fixed, httbar, IOV, text, signal, variable, Linear, usePull, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
                
#             savefilename = f'images/png/{dirname}/{IOV}/{variable}_{cat}{linearStr}.png'
            if usePull:
                savefilename = f'images/png/{dirname}/{IOV}/{variable}_with_Background_and_{signal}_{cat}{linearStr}.png'
            else:
                savefilename = f'images/png/{dirname}/{IOV}/{variable}_{cat}{linearStr}.png'
                
            print(savefilename)
            plt.savefig(savefilename)
            plt.savefig(savefilename.replace('png', 'pdf'))

            plt.show()


In [ ]:
# Signals = {
#     'ZPrime1' : ['1000', '2000', '3000', '4000'],
#     'ZPrime10': ['1000', '2000', '3000', '4000'],
#     'ZPrime30': ['1000', '2000', '3000', '4000'],
#     'ZPrimeDM': ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
#     'RSGluon':  ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000']
# }

# signal = 'ZPrime30'

# IOV = '2018'

# cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']
# cat_labels = ['cen0b', 'fwd0b', 'cen1b', 'fwd1b', 'cen2b', 'fwd2b']

# systematics = ['nominal', 'jes', 'jer', 'pileup', 'pdf', 'q2', 'btag']#, 'prefiring']
# syst_labels = ['nominal']
# for s in systematics:
#     if not 'nominal' in s:
#         syst_labels.append(s+'Down')
#         syst_labels.append(s+'Up')
        
# print(syst_labels)

# savefileheader = '../outputs/combine/categories/TTbarAllHad{}_'.format(IOV.replace('20', '').replace('all',''))
# print(savefileheader)

# froot = uproot.recreate(savefileheader+'CombineRoot_Cat.root')

# variable = 'ttbarmass'



# # for cat, catname in zip(cats, cat_labels):
# #     signal_cat = label_to_int_dict['2t'+cat]
# #     pretag_cat = label_to_int_dict['pret'+cat]
# #     httbar_trial = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat})
# #     print(httbar_trial)



# # for IOV in IOVs:

# for cat, catname in zip(cats, cat_labels):

#     signal_cat = label_to_int_dict['2t'+cat]
#     pretag_cat = label_to_int_dict['pret'+cat]

#     hsignal = {}
#     hsignal['ZPrime1'] = {}
#     hsignal['ZPrime10'] = {}
#     hsignal['ZPrime30'] = {}
#     hsignal['ZPrimeDM'] = {}
#     hsignal['RSGluon'] = {}

#     for syst in syst_labels:

#         if 'Full' in IOV:

#             catsystString = catname+'_'+syst

#             httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_17      = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_17     = getHist(variable, 'TTbar', True, '2017', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_18      = getHist(variable, 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_18     = getHist(variable, 'TTbar', True, '2018', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_17 = getHist(variable, sig, False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_18 = getHist(variable, sig, False, '2018', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_apv + hsignal_noapv + hsignal_17 + hsignal_18

#             httbar  = httbar_apv + httbar_noapv + httbar_17 + httbar_18
#             hcontam = hcontam_apv + hcontam_noapv + hcontam_17 + hcontam_18

#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst
#                 hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_17 = getHist(variable, 'JetHT', True, '2017',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_17 = getHist(variable, 'JetHT', False, '2017',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_18 = getHist(variable, 'JetHT', True, '2018',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_18 = getHist(variable, 'JetHT', False, '2018',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

#                 hntmj = hntmj_apv + hntmj_noapv + hntmj_17 + hntmj_18
#                 hdata = hdata_apv + hdata_noapv + hdata_17 + hdata_18
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
            
#         elif '2016all' in IOV:

#             catsystString = catname+'_'+syst

#             httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#             httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_apv + hsignal_noapv

#             httbar  = httbar_apv + httbar_noapv
#             hcontam = hcontam_apv + hcontam_noapv

#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst
#                 hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

#                 hntmj = hntmj_apv + hntmj_noapv
#                 hdata = hdata_apv + hdata_noapv
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
#         else:

#             catsystString = catname+'_'+syst

#             httbar   = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#             hcontam  = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})

#             print('loading signals...')
#             for sig in Signals.keys():
#                 for mass in Signals[sig]:
#                     hsignal_sigmass   = getHist(variable, sig, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                     hsignal[sig][mass] = hsignal_sigmass


#             print('filling root files...')
#             if 'nominal' in syst:
#                 syst = ''
#                 catsystString = catname+syst

#                 hntmj = getHist(variable, 'JetHT', True, IOV,   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                 hdata = getHist(variable, 'JetHT', False, IOV,  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                 hntmj_fixed = hntmj + -1*hcontam
#                 print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                 froot["data_obs_"+catsystString] = hdata
#                 froot["bkgest_"+catsystString] = hntmj_fixed

#             print('TTbar_'+catsystString)
#             froot["TTbar_"+catsystString] = httbar


#             for sig in Signals.keys():

#                 if 'RSGluon' not in sig:
#                     if sig != 'ZPrime1': # ZPrime10, 30 and DM
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else: # ZPrime1
#                         [print(sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-1]+mass+'_'+sig[-1:]+'_'+catsystString] = hsignal[sig][mass]
#                 else: # RSGluon
#                     [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                     for mass in Signals[sig]:
#                         froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#             # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#             dytext = ''
#             if 'cen' in cat:
#                 dytext = r'$\Delta y$ < 1.0'
#             elif 'fwd' in cat:
#                 dytext = r'$\Delta y$ > 1.0'

#             btext = ''
#             if '0b' in cat:
#                 btext = '0 b-tags'
#             elif '1b' in cat:
#                 btext = '1 b-tag'
#             elif '2b' in cat:
#                 btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'

#             plotBackgroundEstimateNoData(variable, hntmj_fixed, httbar, IOV, text, signal, hsignal['ZPrime1']['3000'], hsignal['ZPrime10']['3000'], hsignal['ZPrime30']['3000'], hsignal['ZPrimeDM']['3000'])
#             plt.show()
                    
# froot.close()            
            
            
            